- node types:  PERSON    and    EMAIL
- relationship types:   SEND    and     RECEIVE

In [ ]:
import json
import re
from email.utils import getaddresses, parsedate_to_datetime
from collections import defaultdict, OrderedDict

INPUT_PATH = "enron_emails.json"
OUTPUT_PATH = "enron_graph_labeled.json"

EMAIL_RE = re.compile(r'<?([A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,})>?', re.I)

def parse_addresses(value):
    if not value:
        return []
    pairs = getaddresses([value])
    out = []
    for name, addr in pairs:
        name = (name or "").strip()
        addr = (addr or "").strip().lower()
        m = EMAIL_RE.search(addr) or EMAIL_RE.search(name)
        if m and not addr:
            addr = m.group(1).lower()
        if not name and not addr:
            continue
        out.append((name, addr))
    seen, uniq = set(), []
    for n, a in out:
        key = (n.lower(), a)
        if key not in seen:
            seen.add(key)
            uniq.append((n, a))
    return uniq

def split_x_display_list(raw):
    if not raw:
        return []
    parts = [p.strip() for p in re.split(r'[;,]', raw) if p.strip()]
    merged = []
    buf = ""
    depth = 0
    for p in parts:
        open_count = p.count("<")
        close_count = p.count(">")
        if depth == 0:
            buf = p
        else:
            buf += ", " + p
        depth += open_count - close_count
        if depth <= 0:
            merged.append(buf.strip())
            buf = ""
            depth = 0
    if buf:
        merged.append(buf.strip())
    return merged

def pair_names_with_emails(email_list, xname_list):
    paired = []
    for i, (n_email, e) in enumerate(email_list):
        n_x = xname_list[i] if i < len(xname_list) else ""
        final_name = n_x if n_x else n_email
        paired.append((final_name, e))
    return paired

def stable_person_key(email, name):
    if email:
        return f"person:{email}"
    base = re.sub(r'[^a-z0-9]+', '_', (name or '').strip().lower()) or "unknown"
    return f"person:name:{base}"

def stable_email_key(message_id, fallback_seed):
    if message_id:
        mid = message_id.strip().strip("<>").strip()
        if mid:
            return f"email:{mid}"
    return "email:derived:" + str(abs(hash(fallback_seed)))

def parse_iso_date(raw):
    if not raw:
        return ""
    try:
        return parsedate_to_datetime(raw).isoformat()
    except Exception:
        return raw

with open(INPUT_PATH, "r", encoding="utf-8") as f:
    messages = json.load(f)

persons = OrderedDict()
emails = OrderedDict()
relationships_tmp = []
send_count = defaultdict(int)
recv_count = defaultdict(int)

for msg in messages:
    h = msg.get("headers", {})
    body = msg.get("body", "")
    date_iso = parse_iso_date(h.get("Date", ""))
    subject = h.get("Subject", "") or ""

    xfrom_names = split_x_display_list((h.get("X-From") or "").strip())
    from_list = parse_addresses(h.get("From", ""))
    sender_name = xfrom_names[0] if xfrom_names else (from_list[0][0] if from_list else "")
    sender_email = from_list[0][1] if from_list else ""
    sender_key = stable_person_key(sender_email, sender_name)

    to_emails  = parse_addresses(h.get("To", ""))
    cc_emails  = parse_addresses(h.get("Cc", "") or h.get("CC", ""))
    bcc_emails = parse_addresses(h.get("Bcc", "") or h.get("BCC", ""))

    x_to_names  = split_x_display_list((h.get("X-To")  or "").strip())
    x_cc_names  = split_x_display_list((h.get("X-cc")  or "").strip())
    x_bcc_names = split_x_display_list((h.get("X-bcc") or "").strip())

    to_recipients  = pair_names_with_emails(to_emails,  x_to_names)
    cc_recipients  = pair_names_with_emails(cc_emails,  x_cc_names)
    bcc_recipients = pair_names_with_emails(bcc_emails, x_bcc_names)

    msg_id_raw = h.get("Message-ID")
    fallback_seed = (h.get("Date","") + "|" + h.get("From","") + "|" + h.get("To","") +
                     "|" + h.get("Cc","") + "|" + h.get("Bcc","") + "|" + subject + "|" + body)
    email_key = stable_email_key(msg_id_raw, fallback_seed)

    emails[email_key] = {
        "type": "email",
        "properties": {
            "date": date_iso,
            "from": sender_email or sender_name,
            "to": ", ".join([e if e else n for n, e in to_recipients]) if to_recipients else "",
            "subject": subject,
            "body": body
        }
    }

    if sender_key not in persons:
        persons[sender_key] = {
            "type": "person",
            "properties": {
                "name": sender_name,
                "email": sender_email,
                "emails_send": 0,
                "emails_received": 0
            }
        }
    send_count[sender_key] += 1

    relationships_tmp.append({
        "type": "send",
        "src_key": sender_key,
        "dst_key": email_key,
        "properties": {"date": date_iso}
    })

    recipients_all = to_recipients + cc_recipients + bcc_recipients

    seen_recipient_keys = set()

    for rname, remail in recipients_all:
        rkey = stable_person_key(remail, rname)
        if rkey not in persons:
            persons[rkey] = {
                "type": "person",
                "properties": {
                    "name": rname,
                    "email": remail,
                    "emails_send": 0,
                    "emails_received": 0
                }
            }
        if rkey in seen_recipient_keys:
            continue
        seen_recipient_keys.add(rkey)

        recv_count[rkey] += 1
        relationships_tmp.append({
            "type": "receive",
            "src_key": email_key,
            "dst_key": rkey,
            "properties": {"date": date_iso}
        })

for pkey, pdata in persons.items():
    pdata["properties"]["emails_send"] = int(send_count.get(pkey, 0))
    pdata["properties"]["emails_received"] = int(recv_count.get(pkey, 0))

numeric_nodes = []
key_to_num = {}

for key, node in list(persons.items()) + list(emails.items()):
    node_id = len(numeric_nodes)
    key_to_num[key] = node_id
    numeric_nodes.append({
        "node_id": node_id,
        "type": node["type"],
        "label": 0,
        "properties": node["properties"]
    })

numeric_rels = []
for ridx, r in enumerate(relationships_tmp):
    numeric_rels.append({
        "rel_id": ridx,
        "type": r["type"],
        "src_id": key_to_num[r["src_key"]],
        "dst_id": key_to_num[r["dst_key"]],
        "properties": r["properties"]
    })

graph = {"nodes": numeric_nodes, "relationships": numeric_rels}

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(graph, f, indent=2, ensure_ascii=False)

print(f"Graph saved to {OUTPUT_PATH}")
print(f"Nodes: {len(numeric_nodes)} | Relationships: {len(numeric_rels)}")

## CLEAN NAMES

In [ ]:
import json
import re

INPUT_PATH = "enron_graph_labeled.json"
OUTPUT_PATH = "enron_graph_labeled_cleaned.json"

CLEAN_RE = re.compile(r'\s*<[^>]*>.*$', re.UNICODE)

def clean_name(name: str) -> str:
    if not isinstance(name, str) or not name.strip():
        return name
    cleaned = CLEAN_RE.sub("", name)
    cleaned = cleaned.replace('"', '').replace("'", "")
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()
    return cleaned

with open(INPUT_PATH, "r", encoding="utf-8") as f:
    graph = json.load(f)

count = 0
for node in graph.get("nodes", []):
    if node.get("type") == "person":
        props = node.get("properties", {})
        name = props.get("name", "")
        cleaned = clean_name(name)
        if cleaned != name:
            props["name"] = cleaned
            count += 1

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(graph, f, indent=2, ensure_ascii=False)

print(f"Cleaned {count} person names and saved to {OUTPUT_PATH}")

# POI - PERSON OF INTEREST

https://github.com/AdmcCarthy/Identify_Fraud_from_Enron_Email/tree/master?tab=readme-ov-file#identify-fraud-from-enron-email

POI = 1 (fraud‐relevant) or POI = 0 (not)

https://github.com/AdmcCarthy/Identify_Fraud_from_Enron_Email/blob/master/resources/other/poi_email_addresses.py#L2

NASTAVILA SEM LABEL = 1 VSEM TEM LJUDEM, KI SO V TEM DATASETU OZNAČENI KOT POI

In [ ]:
def poiEmails():
    email_list = ["kenneth_lay@enron.net",
                  "kenneth_lay@enron.com",
                  "klay.enron@enron.com",
                  "kenneth.lay@enron.com",
                  "klay@enron.com",
                  "layk@enron.com",
                  "chairman.ken@enron.com",
                  "jeffreyskilling@yahoo.com",
                  "jeff_skilling@enron.com",
                  "jskilling@enron.com",
                  "effrey.skilling@enron.com",
                  "skilling@enron.com",
                  "jeffrey.k.skilling@enron.com",
                  "jeff.skilling@enron.com",
                  "kevin_a_howard.enronxgate.enron@enron.net",
                  "kevin.howard@enron.com",
                  "kevin.howard@enron.net",
                  "kevin.howard@gcm.com",
                  "michael.krautz@enron.com",
                  "scott.yeager@enron.com",
                  "syeager@fyi-net.com",
                  "scott_yeager@enron.net",
                  "syeager@flash.net",
                  "joe'.'hirko@enron.com",
                  "joe.hirko@enron.com",
                  "rex.shelby@enron.com",
                  "rex.shelby@enron.nt",
                  "rex_shelby@enron.net",
                  "jbrown@enron.com",
                  "james.brown@enron.com",
                  "rick.causey@enron.com",
                  "richard.causey@enron.com",
                  "rcausey@enron.com",
                  "calger@enron.com",
                  "chris.calger@enron.com",
                  "christopher.calger@enron.com",
                  "ccalger@enron.com",
                  "tim_despain.enronxgate.enron@enron.net",
                  "tim.despain@enron.com",
                  "kevin_hannon@enron.com",
                  "kevin'.'hannon@enron.com",
                  "kevin_hannon@enron.net",
                  "kevin.hannon@enron.com",
                  "mkoenig@enron.com",
                  "mark.koenig@enron.com",
                  "m..forney@enron.com",
                  "ken'.'rice@enron.com",
                  "ken.rice@enron.com",
                  "ken_rice@enron.com",
                  "ken_rice@enron.net",
                  "paula.rieker@enron.com",
                  "prieker@enron.com",
                  "andrew.fastow@enron.com",
                  "lfastow@pdq.net",
                  "andrew.s.fastow@enron.com",
                  "lfastow@pop.pdq.net",
                  "andy.fastow@enron.com",
                  "david.w.delainey@enron.com",
                  "delainey.dave@enron.com",
                  "'delainey@enron.com",
                  "david.delainey@enron.com",
                  "'david.delainey'@enron.com",
                  "dave.delainey@enron.com",
                  "delainey'.'david@enron.com",
                  "ben.glisan@enron.com",
                  "bglisan@enron.com",
                  "ben_f_glisan@enron.com",
                  "jeff.richter@enron.com",
                  "jrichter@nwlink.com",
                  "lawrencelawyer@aol.com",
                  "lawyer'.'larry@enron.com",
                  "larry_lawyer@enron.com",
                  "llawyer@enron.com",
                  "larry.lawyer@enron.com",
                  "lawrence.lawyer@enron.com",
                  "tbelden@enron.com",
                  "tim.belden@enron.com",
                  "tim_belden@pgn.com",
                  "tbelden@ect.enron.com",
                  "michael.kopper@enron.com",
                  "dave.duncan@enron.com",
                  "dave.duncan@cipco.org",
                  "duncan.dave@enron.com",
                  "ray.bowen@enron.com",
                  "raymond.bowen@enron.com",
                  "wes.colwell@enron.com",
                  "dan.boyle@enron.com",
                  "cloehr@enron.com",
                  "chris.loehr@enron.com"
                  ]
    return email_list

In [ ]:
import json
import re

def label_poi_nodes(graph_path="enron_graph_labeled_cleaned.json",
                    output_path="enron_graph_labeled_poi.json"):
    poi_emails_raw = poiEmails()
    poi_emails = set(e.strip().lower() for e in poi_emails_raw if e and isinstance(e, str))

    with open(graph_path, "r", encoding="utf-8") as f:
        graph = json.load(f)

    found_emails = set()
    updated = 0

    for node in graph.get("nodes", []):
        if node.get("type") == "person":
            email = (node.get("properties", {}).get("email") or "").strip().lower()
            if email in poi_emails:
                node["label"] = 1
                found_emails.add(email)
                updated += 1

    missing_emails = sorted(list(poi_emails - found_emails))
    if missing_emails:
        print("POI emails not found:")
        for e in missing_emails:
            print("  -", e)
    else:
        print("All POI emails matched")

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(graph, f, indent=2, ensure_ascii=False)

    print(f"\nUpdated nodes: {updated}")
    print(f"Graph saved to: {output_path}")

label_poi_nodes(
    graph_path="enron_graph_labeled_cleaned.json",
    output_path="enron_graph_labeled_poi.json"
)

In [ ]:
import json

graph_path = "enron_graph_labeled_poi.json"

with open(graph_path, "r", encoding="utf-8") as f:
    graph = json.load(f)

nodes = graph.get("nodes", [])

count_label_0 = sum(1 for n in nodes if n.get("label", 0) == 0)
count_label_1 = sum(1 for n in nodes if n.get("label", 0) == 1)

print(f"Nodes with label 0: {count_label_0}")
print(f"Nodes with label 1: {count_label_1}")
print(f"Total nodes: {len(nodes)}")
